# 05 — App Prototype & Forecast Logic

**Purpose of this notebook**

This notebook bridges the modelling work (notebooks 01–04) and the final Streamlit dashboard (`app.py`). Before building a multi-page web app it is good practice to prototype the core forecast logic in a notebook — you can inspect intermediate results, tweak chart code, and confirm the model behaves correctly without restarting a web server on every change.

**What we test here**

1. Load the champion model (`models/best_model.pkl`) and confirm its identity.
2. Reproduce the test-window forecast (Jan – Mar 2014) and compare to actuals.
3. Extend the forecast horizon beyond the data (1, 7, 14, 30 days).
4. Build the three charts that end up in `app.py`: forecast with CI, residuals over time, residual histogram.
5. Verify all five evaluation metrics match the values reported in notebook 04.

Everything confirmed here is then copy-adapted into `app.py` with `@st.cache_data` / `@st.cache_resource` wrappers added.

## 1. Setup

In [1]:
import os
import sys
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Make src/ importable from notebooks/
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, 'src'))
from evaluation import evaluate_model

# Paths
DATA_PATH   = os.path.join(ROOT, 'data', 'timeseries_features.csv')
MODEL_PATH  = os.path.join(ROOT, 'models', 'best_model.pkl')
PARAMS_PATH = os.path.join(ROOT, 'models', 'phase4_best_params.json')
NAME_PATH   = os.path.join(ROOT, 'models', 'best_model_name.txt')

# Train/test boundary
TRAIN_END  = '2013-12-31'
TEST_START = '2014-01-01'
TEST_END   = '2014-03-31'

print('Setup complete.')

Setup complete.


## 2. Load champion model and metadata

In [2]:
# Load model
with open(MODEL_PATH, 'rb') as f:
    model = pickle.load(f)

# Load champion name
with open(NAME_PATH) as f:
    champion_name = f.read().strip()

# Load best hyperparameters
with open(PARAMS_PATH) as f:
    best_params = json.load(f)

print(f'Champion model : {champion_name}')
print(f'Model type     : {type(model).__name__}')
print(f'Extra regressors: {list(model.extra_regressors.keys()) if hasattr(model, "extra_regressors") else "none"}')
print()
print('Best hyperparameters (Prophet):')
for k, v in best_params.get('prophet', {}).items():
    print(f'  {k}: {v}')

Champion model : Prophet (baseline)
Model type     : Prophet
Extra regressors: ['store_open']

Best hyperparameters (Prophet):
  changepoint_prior_scale: 0.00107950323475759
  holidays_prior_scale: 9.983294804156207
  seasonality_mode: additive
  seasonality_prior_scale: 0.46074622610969335


## 3. Load data

In [3]:
df = pd.read_csv(DATA_PATH, parse_dates=['date']).sort_values('date').reset_index(drop=True)

train = df[df['date'] <= TRAIN_END]
test  = df[(df['date'] >= TEST_START) & (df['date'] <= TEST_END)]

print(f'Full dataset : {len(df)} rows  ({df["date"].min().date()} → {df["date"].max().date()})')
print(f'Training set : {len(train)} rows')
print(f'Test set     : {len(test)} rows')
df.head(3)

Full dataset : 454 rows  (2013-01-02 → 2014-03-31)
Training set : 364 rows
Test set     : 90 rows


,date,unit_sales,store_open,dow,dow_name,month,week,is_weekend,is_national_holiday,dcoilwtico
0,2013-01-02,582.0,1,2,Wednesday,1,1,0,0,93.14
1,2013-01-03,310.0,1,3,Thursday,1,1,0,0,92.97
2,2013-01-04,338.0,1,4,Friday,1,1,0,0,93.12


## 4. Generate forecast

We use `make_future_dataframe(periods = test_rows + horizon)` which returns predictions for:
- All training dates (in-sample fit)
- The 90-day test window
- Any extra horizon days requested

Predictions are clipped to zero — negative sales are impossible.

In [4]:
def make_forecast(model, df, horizon=0):
    """Return Prophet forecast DataFrame covering train + test + horizon days."""
    test_rows = int((df['date'] > TRAIN_END).sum())
    future = model.make_future_dataframe(periods=test_rows + horizon)

    # Inject store_open regressor if the model was trained with it
    if hasattr(model, 'extra_regressors') and 'store_open' in model.extra_regressors:
        store_map = df.set_index('date')['store_open'].to_dict()
        future['store_open'] = future['ds'].map(store_map).fillna(1).astype(int)

    fc = model.predict(future)
    for col in ('yhat', 'yhat_lower', 'yhat_upper'):
        fc[col] = fc[col].clip(lower=0)
    return fc


# Test with no extra horizon first
fc = make_forecast(model, df, horizon=0)
print(f'Forecast rows : {len(fc)}')
print(f'Date range    : {fc["ds"].min().date()} → {fc["ds"].max().date()}')
fc[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(5)

Forecast rows : 454
Date range    : 2013-01-02 → 2014-03-31


,ds,yhat,yhat_lower,yhat_upper
449,2014-03-27,345.716893,205.156994,480.376701
450,2014-03-28,383.188858,246.583712,520.515645
451,2014-03-29,614.681845,467.733586,758.479476
452,2014-03-30,686.704455,544.740305,847.242817
453,2014-03-31,432.980280,293.203042,568.342109


## 5. Evaluate on test window

We use the shared `evaluate_model()` from `src/evaluation.py` — the same function used in every notebook — so the numbers are directly comparable.

In [5]:
fc_idx    = fc.set_index('ds')
test_pred = fc_idx.loc[fc_idx.index.isin(test['date']), 'yhat']
test_act  = test.set_index('date')['unit_sales']
common    = test_pred.index.intersection(test_act.index)

metrics = evaluate_model(
    test_act.loc[common].values,
    test_pred.loc[common].values,
    champion_name,
    fit_time=0.06,
)

print('Test-window metrics')
print('-------------------')
for k, v in metrics.items():
    print(f'{k:<15} {v}')

# Confirm these match notebook 04
assert abs(metrics['MAE']  - 94.9) < 1.0, 'MAE mismatch vs notebook 04!'
assert abs(metrics['R²']   - 0.437) < 0.01, 'R² mismatch vs notebook 04!'
print('\n✅  Metrics match notebook 04 — champion model loaded correctly.')

Test-window metrics
-------------------
Model           Prophet (baseline)
MAE             94.9
RMSE            142.8
MAPE (%)        21.0
Bias            -10.6
R²              0.437
Fit time (s)    0.06

✅  Metrics match notebook 04 — champion model loaded correctly.


## 6. Test extended horizon forecasts

The Streamlit app lets users extend the forecast up to 30 days beyond Mar 31 2014. We confirm that logic works here before wiring it up in the app.

In [6]:
for horizon in [1, 7, 14, 30]:
    fc_ext = make_forecast(model, df, horizon=horizon)
    last_date = fc_ext['ds'].max()
    last_pred = fc_ext['yhat'].iloc[-1]
    print(f'Horizon +{horizon:2d} d → last forecast date: {last_date.date()}  yhat: {last_pred:.1f}')

Horizon + 1 d → last forecast date: 2014-04-01  yhat: 380.9
Horizon + 7 d → last forecast date: 2014-04-07  yhat: 432.6
Horizon +14 d → last forecast date: 2014-04-14  yhat: 432.2
Horizon +30 d → last forecast date: 2014-04-30  yhat: 448.1


## 7. Chart prototypes

The three Plotly charts built below are the exact charts used in `app.py → page_forecast()`. Testing them here first ensures axis labels, color choices, and layout parameters are correct before they go into the web app.

### 7.1 Main forecast chart

In [7]:
ACCENT = '#7C3AED'
GREEN  = '#10B981'
CYAN   = '#06B6D4'

fc_30 = make_forecast(model, df, horizon=14)

fig = go.Figure()

# Training data
fig.add_trace(go.Scatter(
    x=train['date'], y=train['unit_sales'],
    name='Training data', mode='lines',
    line=dict(color='#4B5563', width=1.2), opacity=0.6,
))

# Test actuals
fig.add_trace(go.Scatter(
    x=test['date'], y=test['unit_sales'],
    name='Actuals (test)', mode='lines+markers',
    line=dict(color=GREEN, width=2),
    marker=dict(size=4),
))

# Confidence interval
fig.add_trace(go.Scatter(
    x=pd.concat([fc_30['ds'], fc_30['ds'][::-1]]),
    y=pd.concat([fc_30['yhat_upper'], fc_30['yhat_lower'][::-1]]),
    fill='toself', fillcolor='rgba(124,58,237,0.10)',
    line=dict(color='rgba(0,0,0,0)'), name='80 % CI',
))

# Forecast line
fig.add_trace(go.Scatter(
    x=fc_30['ds'], y=fc_30['yhat'],
    name='Prophet forecast', mode='lines',
    line=dict(color=ACCENT, width=2.2, dash='dot'),
))

# Train/test split
fig.add_shape(type='line', x0=TRAIN_END, x1=TRAIN_END, y0=0, y1=1, yref='paper',
              line=dict(dash='dash', color='#6B7280', width=1.5))
fig.add_annotation(x=TRAIN_END, y=0.97, yref='paper', text='Train / Test split',
                   showarrow=False, xanchor='left', xshift=6,
                   font=dict(color='#9CA3AF', size=11))

# Future shading
fut_start = pd.Timestamp(TEST_END) + pd.Timedelta(days=1)
fig.add_vrect(x0=fut_start, x1=fc_30['ds'].max(),
              fillcolor='rgba(6,182,212,0.07)', line_width=0)
fig.add_annotation(x=fut_start, y=0.97, yref='paper', text='+14 d forecast',
                   showarrow=False, xanchor='left', xshift=6,
                   font=dict(color=CYAN, size=11))

fig.update_layout(
    template='plotly_dark',
    title='Prophet Forecast — Champion Model',
    xaxis_title='Date', yaxis_title='Unit Sales',
    height=460,
    legend=dict(orientation='h', y=1.1, xanchor='right', x=1),
)
fig.show()

### 7.2 Residual charts

In [8]:
residuals = test_act.loc[common] - test_pred.loc[common]

# Residuals over time
fig_r = go.Figure()
fig_r.add_trace(go.Scatter(
    x=common, y=residuals,
    mode='lines+markers',
    line=dict(color=ACCENT, width=1.8),
    marker=dict(size=4),
    name='Residual',
))
fig_r.add_hline(y=0, line_dash='dash', line_color='#6B7280')
fig_r.update_layout(
    template='plotly_dark',
    title='Residuals over time (Actual − Predicted)',
    xaxis_title='Date', yaxis_title='Residual',
    height=320,
)
fig_r.show()

# Residual histogram
fig_h = go.Figure()
fig_h.add_trace(go.Histogram(
    x=residuals, nbinsx=20,
    marker_color=ACCENT, opacity=0.85,
))
fig_h.update_layout(
    template='plotly_dark',
    title='Residual distribution',
    xaxis_title='Residual value', yaxis_title='Count',
    height=320,
)
fig_h.show()

print(f'Mean residual (Bias): {residuals.mean():.1f}')
print(f'Std  residual       : {residuals.std():.1f}')
print(f'Min                 : {residuals.min():.1f}')
print(f'Max                 : {residuals.max():.1f}')

Mean residual (Bias): 10.6
Std  residual       : 143.2
Min                 : -279.2
Max                 : 819.2


## 8. Summary

| Check | Result |
|---|---|
| Model loads correctly | ✅ |
| Test-window metrics match notebook 04 | ✅ |
| Extended horizons (1, 7, 14, 30 d) | ✅ |
| Forecast chart renders | ✅ |
| Residual charts render | ✅ |

All logic confirmed. The `make_forecast()` function and chart code above are the direct source of `app.py → page_forecast()`. The Streamlit app wraps each section with `@st.cache_data` / `@st.cache_resource` and adds sidebar controls for the horizon slider and display toggles.